# RavenPack Macro News Extraction — Data Collection

**Academic research only — not investment advice.** Output of this project is a comparative evaluation of LLM-derived sentiment vs. market features for SP500 sector ETFs; nothing here is a trading signal.

This notebook is the data-engineering (silver/gold) counterpart to `Basic_EDA_Analysis.ipynb`: it re-runs the same RavenPack filtering logic but as a clean, reusable extraction pipeline rather than an exploratory notebook. It produces two outputs:

1. **Silver — core filtered event table** (`data_collection/raw/ravenpack_core_events_<START>_<END>.csv`): row-level RavenPack macro events, 2020–2025, filtered to rank-1 non-blog sources and relevance/event_relevance ≥ 90, unioned across years into one table.
2. **Gold — daily summary table** (`news_daily_df`, saved to `data_collection/news_daily_df.csv`): one row per NYSE trading session, aggregating the silver table with the after-hours (4:00 PM ET) lookahead-bias cutoff already applied.

**Licensing note:** WRDS RavenPack is a licensed academic dataset. The silver output is row-level and is written to `data_collection/raw/`, which is `.gitignore`'d — it must never be committed. Only the aggregated gold table (`news_daily_df.csv`) is safe to commit.

In [1]:
%pip install -q wrds psycopg2-binary pandas numpy pyarrow pandas_market_calendars

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\jerem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT
    NOTEBOOK_DIR = REPO_ROOT / "data_collection"
elif PROJECT_ROOT.name == "data_collection" and (PROJECT_ROOT.parent / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT.parent
    NOTEBOOK_DIR = PROJECT_ROOT
else:
    raise FileNotFoundError("Run this notebook from the repository root or data_collection/.")
RAW_DIR = NOTEBOOK_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_DATE = "2020-01-01"
END_DATE = "2025-12-31"
YEARS = range(2020, 2026)

RELEVANCE_MIN = 90
EVENT_RELEVANCE_MIN = 90
MARKET_CLOSE_ET = "16:00:00"

CORE_EVENTS_CSV = RAW_DIR / f"ravenpack_core_events_{START_DATE[:4]}_{END_DATE[:4]}.csv"
NEWS_DAILY_CSV = NOTEBOOK_DIR / "news_daily_df.csv"

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Silver output (gitignored, raw/):  {CORE_EVENTS_CSV}")
print(f"Gold output (committed, aggregated): {NEWS_DAILY_CSV}")

Date range: 2020-01-01 to 2025-12-31
Silver output (gitignored, raw/):  d:\Documents\MADS MICH\SIADS 699 Capstone\siads-699-sentiment-analysis-sp500\data_collection\raw\ravenpack_core_events_2020_2025.csv
Gold output (committed, aggregated): d:\Documents\MADS MICH\SIADS 699 Capstone\siads-699-sentiment-analysis-sp500\data_collection\news_daily_df.csv


In [4]:
# Connect to WRDS. This may prompt for credentials if no .pgpass file is configured.
db = wrds.Connection()

rp_tables = db.list_tables(library="rpna")
expected_rp_tables = [f"rpa_djpr_global_macro_{year}" for year in YEARS]
missing_rp_tables = [table for table in expected_rp_tables if table not in rp_tables]

if missing_rp_tables:
    raise RuntimeError(f"Missing RavenPack tables: {missing_rp_tables}")

print("WRDS connection ready.")
print("RavenPack macro tables found:", expected_rp_tables)

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\jerem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
WRDS connection ready.
RavenPack macro tables found: ['rpa_djpr_global_macro_2020', 'rpa_djpr_global_macro_2021', 'rpa_djpr_global_macro_2022', 'rpa_djpr_global_macro_2023', 'rpa_djpr_global_macro_2024', 'rpa_djpr_global_macro_2025']


## Schema check: does a full-text column exist?

RavenPack's WRDS feed is licensed as sentiment/metadata analytics, not full-article redistribution, so the tables used below are not expected to carry a full article body column. Confirm this empirically against the live schema rather than assuming it — column availability can vary by RavenPack product tier.

In [5]:
macro_table_schema_df = db.describe_table("rpna", f"rpa_djpr_global_macro_{max(YEARS)}")
source_list_schema_df = db.describe_table("rpna", "rpa_source_list")

print("Columns in rpa_djpr_global_macro_<year>:")
display(macro_table_schema_df)

print("Columns in rpa_source_list:")
display(source_list_schema_df)

# Flags any column name that looks like it could hold free text (headline/body/summary/etc.)
text_like_columns = macro_table_schema_df.loc[
    macro_table_schema_df["name"].str.contains("headline|title|text|body|summary|content", case=False, na=False)
]
print("Columns that look like they might hold article text or headlines:")
display(text_like_columns)

Approximately 21214124 rows in rpna.rpa_djpr_global_macro_2025.
Approximately 103851 rows in rpna.rpa_source_list.
Columns in rpa_djpr_global_macro_<year>:


,name,nullable,type,comment
0,rpa_date_utc,True,DATE,None
1,rpa_time_utc,True,TIME,None
2,timestamp_utc,True,TIMESTAMP,None
3,rp_story_id,True,VARCHAR(32),None
4,rp_entity_id,True,VARCHAR(6),None
5,entity_type,True,VARCHAR(4),None
6,entity_name,True,VARCHAR(400),None
7,country_code,True,VARCHAR(2),None
8,relevance,True,DOUBLE PRECISION,None
9,event_sentiment_score,True,DOUBLE PRECISION,None


Columns in rpa_source_list:


,name,nullable,type,comment
0,rp_entity_id,True,VARCHAR(6),None
1,entity_type,True,VARCHAR(4),None
2,data_type,True,VARCHAR(20),None
3,data_value,True,VARCHAR(400),None
4,range_start,True,DATE,None
5,range_end,True,DATE,None


Columns that look like they might hold article text or headlines:


,name,nullable,type,comment
32,event_text,True,VARCHAR(400),None
51,headline,True,VARCHAR(4000),None


### Peek at actual `headline` / `event_text` values

Small, unfiltered-by-source sample so you can see real content and typical string lengths before deciding whether/how to bring these columns into the silver table. This output is not saved to disk — it is for interactive inspection only.

In [ ]:
# Do not query or display licensed article text or headlines.
# The current deliverable uses RavenPack's structured sentiment fields only.
print("Raw headline/event_text inspection skipped by design.")

## 1. Build the institutional source universe

Rank-1, non-blog sources only, same filter as `Basic_EDA_Analysis.ipynb`.

In [ ]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


source_query = """
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
"""

raw_source_attributes = db.raw_sql(source_query)

source_attribute_dupes = raw_source_attributes.duplicated(["rp_entity_id", "data_type"]).sum()
if source_attribute_dupes:
    raise ValueError(f"Source metadata has {source_attribute_dupes:,} duplicate entity/data_type rows; resolve before pivoting.")

sources_wide = raw_source_attributes.pivot(
    index="rp_entity_id", columns="data_type", values="data_value"
).reset_index()
sources_wide.columns.name = None

sources_df = sources_wide.rename(columns={
    "ENTITY_NAME": "source_name",
    "PUBLICATION_TYPE": "source_type",
    "SOURCE_RANK": "source_rank",
})

sources_df["source_rank"] = pd.to_numeric(sources_df["source_rank"], errors="coerce")
sources_df["is_rank1_non_blog"] = (
    sources_df["source_rank"].eq(1)
    & sources_df["source_type"].notna()
    & sources_df["source_type"].ne("BLOG")
)

institutional_sources_df = sources_df.loc[sources_df["is_rank1_non_blog"]].copy()
valid_source_ids = institutional_sources_df["rp_entity_id"].dropna().astype(str).tolist()
source_id_sql = sql_string_list(valid_source_ids)

print(f"All RavenPack source entities: {len(sources_df):,}")
print(f"Rank-1 non-blog institutional sources: {len(institutional_sources_df):,}")
display(institutional_sources_df.head())

## 2. Extract & union core filtered RavenPack events, 2020–2025 (Silver)

Row-level macro events from `rpna.rpa_djpr_global_macro_<year>`, filtered to relevance/event_relevance ≥ 90 and rank-1 non-blog sources, with the after-hours (4:00 PM ET) lookahead cutoff computed directly in SQL as `signal_calendar_date`. One year at a time, then unioned into a single table.

Includes `headline` and `event_text` so the NLP/LLM lead can run our own LLM sentiment scoring on the same text RavenPack scored, then compare our LLM-derived scores against `event_sentiment_score` (RavenPack's built-in score). These two columns are real article text, so this reinforces why the silver table stays under the gitignored `data_collection/raw/` path.

In [ ]:
ET_TIMESTAMP_SQL = "((timestamp_utc AT TIME ZONE 'UTC') AT TIME ZONE 'America/New_York')"


def fetch_core_events_for_year(year):
    table_name = f"rpna.rpa_djpr_global_macro_{year}"
    query = f"""
        SELECT
            rp_story_id,
            timestamp_utc,
            CASE
                WHEN {ET_TIMESTAMP_SQL}::time < TIME '{MARKET_CLOSE_ET}'
                    THEN {ET_TIMESTAMP_SQL}::date
                ELSE ({ET_TIMESTAMP_SQL}::date + INTERVAL '1 day')::date
            END AS signal_calendar_date,
            relevance,
            event_relevance,
            rp_source_id,
            source_name,
            topic,
            "group" AS group_name,
            event_sentiment_score,
            headline,
            event_text
        FROM {table_name}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance >= {RELEVANCE_MIN}
          AND event_relevance >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND timestamp_utc IS NOT NULL
          AND event_sentiment_score IS NOT NULL
    """
    df = db.raw_sql(query)
    df["source_year"] = year
    return df


core_event_parts = []
for year in YEARS:
    print(f"Extracting core RavenPack macro events for {year}...")
    core_event_parts.append(fetch_core_events_for_year(year))

news_events_df = pd.concat(core_event_parts, ignore_index=True)

news_events_df["timestamp_utc"] = pd.to_datetime(news_events_df["timestamp_utc"])
news_events_df["signal_calendar_date"] = pd.to_datetime(news_events_df["signal_calendar_date"])
news_events_df["relevance"] = pd.to_numeric(news_events_df["relevance"], errors="coerce")
news_events_df["event_relevance"] = pd.to_numeric(news_events_df["event_relevance"], errors="coerce")
news_events_df["event_sentiment_score"] = pd.to_numeric(news_events_df["event_sentiment_score"], errors="coerce")

news_events_df = news_events_df.drop_duplicates(subset=["rp_story_id", "timestamp_utc", "rp_source_id"])

print(f"Core filtered event rows (2020-2025, unioned): {len(news_events_df):,}")
print(f"Distinct stories: {news_events_df['rp_story_id'].nunique():,}")
print(f"Rows with non-null headline: {news_events_df['headline'].notna().sum():,}")
print(f"Rows with non-null event_text: {news_events_df['event_text'].notna().sum():,}")
display(news_events_df.head())

In [ ]:
# Licensed WRDS data — write only to the gitignored raw/ folder, never to a tracked path.
news_events_df.to_csv(CORE_EVENTS_CSV, index=False)
print(f"Saved silver (row-level) table to: {CORE_EVENTS_CSV}")
print("This path is under data_collection/raw/, which is .gitignore'd - do not force-add it.")

## 3. Infer the RavenPack sentiment scale

Computed directly from the unioned silver table (no need to re-query WRDS), same logic as `Basic_EDA_Analysis.ipynb`.

In [ ]:
score_sample = news_events_df["event_sentiment_score"].dropna()

if score_sample.empty:
    raise RuntimeError("No event_sentiment_score values found in the unioned core events table.")

score_min = float(score_sample.min())
score_max = float(score_sample.max())

if score_min >= -1.5 and score_max <= 1.5:
    SENTIMENT_NEGATIVE_MAX = -0.05
    SENTIMENT_POSITIVE_MIN = 0.05
    SENTIMENT_SCALE_NOTE = "approx. -1 to 1 scale"
elif score_min >= 0 and score_max <= 100:
    SENTIMENT_NEGATIVE_MAX = 50.0
    SENTIMENT_POSITIVE_MIN = 50.0
    SENTIMENT_SCALE_NOTE = "approx. 0 to 100 scale"
else:
    SENTIMENT_NEGATIVE_MAX = -1.0
    SENTIMENT_POSITIVE_MIN = 1.0
    SENTIMENT_SCALE_NOTE = "wide signed scale fallback"

print({
    "rows": len(score_sample),
    "min": score_min,
    "mean": float(score_sample.mean()),
    "max": score_max,
    "scale_note": SENTIMENT_SCALE_NOTE,
    "negative_if_less_than": SENTIMENT_NEGATIVE_MAX,
    "positive_if_greater_than": SENTIMENT_POSITIVE_MIN,
})

## 4. Build the daily aggregated summary table (Gold): `news_daily_df`

Two-stage aggregation, same approach as `Basic_EDA_Analysis.ipynb`:

1. Aggregate the silver table by `signal_calendar_date` (the 4:00 PM ET cutoff already applied).
2. Map each calendar date to the next NYSE trading session using `pandas_market_calendars` (no CRSP dependency needed in this notebook), then re-aggregate to `session_date` so news from non-trading days (weekends/holidays) rolls forward correctly.

In [ ]:
import pandas_market_calendars as mcal

news_events_df["is_positive"] = news_events_df["event_sentiment_score"] > SENTIMENT_POSITIVE_MIN
news_events_df["is_negative"] = news_events_df["event_sentiment_score"] < SENTIMENT_NEGATIVE_MAX
news_events_df["is_neutral"] = ~news_events_df["is_positive"] & ~news_events_df["is_negative"]

calendar_daily_df = (
    news_events_df
    .groupby("signal_calendar_date")
    .agg(
        event_record_count=("rp_story_id", "size"),
        unique_story_count=("rp_story_id", "nunique"),
        positive_event_count=("is_positive", "sum"),
        negative_event_count=("is_negative", "sum"),
        neutral_event_count=("is_neutral", "sum"),
        unique_source_count=("rp_source_id", "nunique"),
        sentiment_sum=("event_sentiment_score", "sum"),
    )
    .reset_index()
)

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=START_DATE, end_date=(pd.Timestamp(END_DATE) + pd.Timedelta(days=7)))
trading_sessions = pd.Series(pd.to_datetime(schedule.index)).sort_values()


def map_to_next_trading_session(dates, sessions):
    session_values = sessions.to_numpy(dtype="datetime64[ns]")
    target_values = pd.to_datetime(dates).to_numpy(dtype="datetime64[ns]")
    positions = np.searchsorted(session_values, target_values, side="left")
    mapped = np.full(len(target_values), np.datetime64("NaT"), dtype="datetime64[ns]")
    valid = positions < len(session_values)
    mapped[valid] = session_values[positions[valid]]
    return pd.to_datetime(mapped)


calendar_daily_df["session_date"] = map_to_next_trading_session(
    calendar_daily_df["signal_calendar_date"], trading_sessions
)
calendar_daily_df = calendar_daily_df.dropna(subset=["session_date"])

news_daily_df = (
    calendar_daily_df
    .groupby("session_date", as_index=False)
    .agg(
        event_record_count=("event_record_count", "sum"),
        unique_story_count=("unique_story_count", "sum"),
        positive_event_count=("positive_event_count", "sum"),
        negative_event_count=("negative_event_count", "sum"),
        neutral_event_count=("neutral_event_count", "sum"),
        unique_source_count=("unique_source_count", "max"),
        sentiment_sum=("sentiment_sum", "sum"),
    )
)

# Recompute source breadth from event rows after session mapping. A max across
# calendar dates can undercount sources when a weekend/holiday rolls forward.
event_session_sources = news_events_df[["signal_calendar_date", "rp_source_id"]].copy()
event_session_sources["session_date"] = map_to_next_trading_session(
    event_session_sources["signal_calendar_date"], trading_sessions
)
source_counts_by_session = (
    event_session_sources.dropna(subset=["session_date"])
    .groupby("session_date")["rp_source_id"]
    .nunique()
)
news_daily_df["unique_source_count"] = (
    news_daily_df["session_date"].map(source_counts_by_session).fillna(0).astype(int)
)

news_daily_df["mean_event_sentiment_score"] = (
    news_daily_df["sentiment_sum"] / news_daily_df["event_record_count"]
)
news_daily_df = news_daily_df.drop(columns=["sentiment_sum"])

for label in ["positive", "negative", "neutral"]:
    news_daily_df[f"{label}_event_share"] = (
        news_daily_df[f"{label}_event_count"] / news_daily_df["event_record_count"]
    )

conditions = [
    news_daily_df["mean_event_sentiment_score"] > SENTIMENT_POSITIVE_MIN,
    news_daily_df["mean_event_sentiment_score"] < SENTIMENT_NEGATIVE_MAX,
]
news_daily_df["sentiment_bucket"] = np.select(conditions, ["positive", "negative"], default="neutral")
news_daily_df = news_daily_df.sort_values("session_date").reset_index(drop=True)

print(f"Trading-session news rows: {len(news_daily_df):,}")
display(news_daily_df.head())

## 5. Validation checks

In [ ]:
assert (news_daily_df["unique_story_count"] <= news_daily_df["event_record_count"]).all(), (
    "unique_story_count should not exceed event_record_count"
)

share_sum = news_daily_df[["positive_event_share", "negative_event_share", "neutral_event_share"]].sum(axis=1)
assert np.allclose(share_sum, 1.0, atol=1e-6), "Sentiment shares do not sum to 1.0"

duplicate_sessions = news_daily_df["session_date"].duplicated().sum()
assert duplicate_sessions == 0, f"Duplicate session_date rows found: {duplicate_sessions}"

print("Validation checks passed.")
print(f"Calendar date range: {news_events_df['signal_calendar_date'].min().date()} to {news_events_df['signal_calendar_date'].max().date()}")
print(f"Session date range:  {news_daily_df['session_date'].min().date()} to {news_daily_df['session_date'].max().date()}")

# Spot-check (visual only): after-hours (>= 4pm ET) articles should roll to the next trading session,
# not the same day - this is the lookahead-bias guard called out in CLAUDE.md.
after_hours_mask = (
    news_events_df["timestamp_utc"].dt.tz_localize("UTC").dt.tz_convert("America/New_York").dt.time
    >= pd.Timestamp(MARKET_CLOSE_ET).time()
)
after_hours_spot_check = news_events_df.loc[
    after_hours_mask, ["timestamp_utc", "signal_calendar_date", "rp_story_id"]
].head(5).copy()
after_hours_spot_check["mapped_session_date"] = map_to_next_trading_session(
    after_hours_spot_check["signal_calendar_date"], trading_sessions
)
display(after_hours_spot_check)

## 6. Save the gold summary table and print a final run summary

`news_daily_df.csv` is aggregated/derived, not raw WRDS records, so it is safe to commit per the licensing constraint in `CLAUDE.md`.

In [ ]:
news_daily_df.to_csv(NEWS_DAILY_CSV, index=False)

print("Extraction complete.")
print(f"Silver (row-level, gitignored):     {CORE_EVENTS_CSV}  [{len(news_events_df):,} rows]")
print(f"Gold (daily summary, committed):     {NEWS_DAILY_CSV}  [{len(news_daily_df):,} rows]")
print(f"Rank-1 non-blog sources used:        {len(institutional_sources_df):,}")
print(f"Sentiment scale detected:            {SENTIMENT_SCALE_NOTE}")